In [2]:
from pathlib import Path
import sys

project_root = Path.cwd().parent
sys.path.append(str(project_root))

In [3]:
from embedder import Embedder

embed = Embedder()

q1 = "How does approximate nearest neighbor search work?"

v1 = embed.encode(q1)


In [4]:
print(v1[0])

-0.02058203437252893


In [5]:
from gitsource import GithubRepositoryDataReader

reader = GithubRepositoryDataReader(
    repo_owner="DataTalksClub",
    repo_name="llm-zoomcamp",
    commit_id="8c1834d",
    allowed_extensions={"md"},
    filename_filter=lambda path: "/lessons/" in path,
)

documents = [file.parse() for file in reader.read()]

In [23]:
print(type(documents))
print(type(documents[0]))
print(documents[0].keys())




<class 'list'>
<class 'dict'>
dict_keys(['content', 'filename'])


In [32]:
doc = None

for d in documents:
    if d["filename"] == "02-vector-search/lessons/07-sqlitesearch-vector.md":
        doc = d["content"]
        break

d1 = embed.encode(doc)

In [33]:
import numpy as np

similarity = np.dot(v1, d1)

print(similarity)

0.36107026789538205


In [68]:
from gitsource import chunk_documents
chunks = chunk_documents(documents, size=2000, step=1000)

In [37]:
chunk_contents = [chunk["content"] for chunk in chunks]
c1 = embed.encode_batch(chunk_contents)

In [43]:
print(c1.shape)
print(v1.shape)

scores = c1.dot(v1)
best_score = scores.argmax()

print(best_score)

best_chunk = chunks[best_score]

print(best_chunk["filename"])
print(scores[best_score])

(295, 384)
(384,)
94
02-vector-search/lessons/07-sqlitesearch-vector.md
0.6489016436447387


In [44]:
print(best_chunk["filename"])

02-vector-search/lessons/07-sqlitesearch-vector.md


In [65]:
from minsearch import VectorSearch

vindex = VectorSearch()
vindex.fit(c1, chunks)

In [66]:
query_vector = embed.encode("What metric do we use to evaluate a search engine?")

results = vindex.search(query_vector=query_vector)

print(type(results))
print(type(results[0]))
print(results[0]["filename"])

<class 'list'>
<class 'dict'>
04-evaluation/lessons/05-search-metrics.md


In [74]:
#text search vs vector search
from minsearch import Index

query = "How do I store vectors in PostgreSQL?"

text_index = Index(
    text_fields=["content"],
    keyword_fields=["filename"]
)

text_index.fit(chunks)

text_results = text_index.search(query, num_results=5)


query_vector = embed.encode(query)

vindex = VectorSearch()
vindex.fit(c1, chunks)

vector_results = vindex.search(
    query_vector=query_vector,
    num_results=5
)

text_files = [r["filename"] for r in text_results]
vector_files = [r["filename"] for r in vector_results]

print("Text search:")
for f in text_files:
    print(f)

print("\nVector search:")
for f in vector_files:
    print(f)

Text search:
02-vector-search/lessons/02-embeddings.md
03-orchestration/lessons/05-rag.md
02-vector-search/lessons/01-intro.md
03-orchestration/lessons/05-rag.md
02-vector-search/lessons/01-intro.md

Vector search:
02-vector-search/lessons/08-pgvector.md
02-vector-search/lessons/08-pgvector.md
03-orchestration/lessons/05-rag.md
02-vector-search/lessons/08-pgvector.md
02-vector-search/lessons/08-pgvector.md


In [75]:
def rrf(result_lists, k=60, num_results=5):
    scores = {}
    docs = {}

    for results in result_lists:
        for rank, doc in enumerate(results):
            key = (doc["filename"], doc["start"])
            scores[key] = scores.get(key, 0) + 1 / (k + rank)
            docs[key] = doc

    ranked = sorted(scores, key=scores.get, reverse=True)
    return [docs[key] for key in ranked[:num_results]]

In [76]:
#Hybrid Search
from minsearch import Index

query = "How do I give the model access to tools?"

# Text search
text_index = Index(
    text_fields=["content"],
    keyword_fields=["filename"]
)

text_index.fit(chunks)

text_results = text_index.search(query, num_results=5)

# Vector search
query_vector = embed.encode(query)

vindex = VectorSearch()
vindex.fit(c1, chunks)

vector_results = vindex.search(
    query_vector=query_vector,
    num_results=5
)

# Hybrid search
results = rrf([vector_results, text_results])

print("Vector:")
for r in vector_results:
    print(r["filename"], r["start"])

print("\nText:")
for r in text_results:
    print(r["filename"], r["start"])

print("\nRRF:")
for r in results:
    print(r["filename"], r["start"])

print("\nAnswer:")
print(results[0]["filename"])

Vector:
01-agentic-rag/lessons/01-intro.md 2000
04-evaluation/lessons/02-ground-truth.md 1000
01-agentic-rag/lessons/16-other-frameworks.md 0
01-agentic-rag/lessons/15-frameworks.md 2000
01-agentic-rag/lessons/13-function-calling.md 4000

Text:
01-agentic-rag/lessons/14-agentic-loop.md 0
01-agentic-rag/lessons/13-function-calling.md 4000
01-agentic-rag/lessons/13-function-calling.md 5000
01-agentic-rag/lessons/13-function-calling.md 1000
04-evaluation/lessons/02-ground-truth.md 3000

RRF:
01-agentic-rag/lessons/13-function-calling.md 4000
01-agentic-rag/lessons/01-intro.md 2000
01-agentic-rag/lessons/14-agentic-loop.md 0
04-evaluation/lessons/02-ground-truth.md 1000
01-agentic-rag/lessons/16-other-frameworks.md 0

Answer:
01-agentic-rag/lessons/13-function-calling.md
